# Weather Destination Finder

A pipeline that builds a global weather dataset, filters destinations by temperature preference, finds nearby hotels, and generates a multi-stop driving itinerary - all visualized on interactive maps. Explores the usage of API's for data wrangling. 

In [ ]:
# If needed: 
# %pip install pandas numpy hvplot geoviews citipy requests

In [ ]:
import numpy as np
import pandas as pd
import requests
import time
import os
import warnings

import hvplot.pandas
import geoviews as gv
from citipy import citipy
from config import weather_api_key, geoapify_key

warnings.filterwarnings("ignore")

---
## 1. Build the Weather Database

We sample 2,000 random latitude/longitude pairs uniformly across the globe, then use `citipy` to find the nearest city for each point. Duplicates are filtered out to avoid redundant API calls.

If the weather CSV already exists from a previous run, we load it directly instead of re-fetching  - this avoids unnecessary API calls and speeds up iteration.

In [4]:
# Define the path for the cached data
DATA_PATH = "Weather_Database/WeatherPy_Database.csv"
DATA_NUM_ROWS = 2000

os.makedirs("data", exist_ok=True) 

# Load if exists, otherwise build the dataset
if os.path.exists(DATA_PATH):
    city_data_df = pd.read_csv(DATA_PATH)
    print(f"Loaded cached weather data - {len(city_data_df)} cities.")
else:
    # Generate 2000 random coordinate pairs
    lats = np.random.uniform(low = -90.0, high = 90.0, size = DATA_NUM_ROWS)
    lngs = np.random.uniform(low = -180.0, high = 180.0, size = DATA_NUM_ROWS)
    coordinates = list(zip(lats, lngs))

    # Map each coordinate to its nearest city (deduplicated)
    cities = []
    for coordinate in coordinates:
        city = citipy.nearest_city(coordinate[0], coordinate[1]).city_name
        if city not in cities:
            cities.append(city)

    print(f"Identified {len(cities)} unique cities from {DATA_NUM_ROWS} random coordinates.")

    # Fetch weather data from OpenWeatherMap
    url = "http://api.openweathermap.org/data/2.5/weather?units=Imperial&APPID=" + weather_api_key

    # Create an empty list to hold weather data for each city
    city_data = []
    record_count = 1
    set_count = 1

    print("-" * 30)
    print("Beginning Data Retrieval")
    print("-" * 30)

    for i, city in enumerate(cities):
        # Pause between batches to respect rate limits
        if i % 50 == 0 and i >= 50:
            set_count += 1
            record_count = 1
            time.sleep(60)

        city_url = url + "&q=" + city.replace(" ", "+")
        print(f"Processing Record {record_count} of Set {set_count} | {city}")
        record_count += 1

        # API request for each of the cities
        try:
            response = requests.get(city_url)
            response.raise_for_status()
            city_weather = response.json()

            city_data.append({
                "City": city.title(),
                "Country": city_weather["sys"]["country"],
                "Lat": city_weather["coord"]["lat"],
                "Lng": city_weather["coord"]["lon"],
                "Max Temp": city_weather["main"]["temp_max"],
                "Humidity": city_weather["main"]["humidity"],
                "Cloudiness": city_weather["clouds"]["all"],
                "Wind Speed": city_weather["wind"]["speed"],
                "Current Description": city_weather["weather"][0]["description"]
            })

        except (requests.exceptions.RequestException, KeyError) as e:
            print(f"  City not found or API error. Skipping... ({e})")
            continue

    print("-" * 30)
    print(f"Data Retrieval Complete  - {len(city_data)} cities collected.")
    print("-" * 30)

    # Build DataFrame and export
    city_data_df = pd.DataFrame(city_data)
    city_data_df = city_data_df[
        ["City", "Country", "Lat", "Lng", "Max Temp",
         "Humidity", "Cloudiness", "Wind Speed", "Current Description"]
    ]
    city_data_df.to_csv(DATA_PATH, index=False)
    print(f"Saved to {DATA_PATH}")

Identified 742 unique cities from 2000 random coordinates.
------------------------------
Beginning Data Retrieval
------------------------------
Processing Record 1 of Set 1 | waitangi
Processing Record 2 of Set 1 | andergrove
Processing Record 3 of Set 1 | port-aux-francais
Processing Record 4 of Set 1 | praia da vitoria
Processing Record 5 of Set 1 | touros
Processing Record 6 of Set 1 | sittwe
Processing Record 7 of Set 1 | saudarkrokur
Processing Record 8 of Set 1 | tura
Processing Record 9 of Set 1 | grytviken
Processing Record 10 of Set 1 | afaahiti
Processing Record 11 of Set 1 | badger
Processing Record 12 of Set 1 | 'ohonua
  City not found or API error. Skipping... (404 Client Error: Not Found for url: http://api.openweathermap.org/data/2.5/weather?units=Imperial&APPID=c86d0e31836a28942761cfac0857646a&q='ohonua)
Processing Record 13 of Set 1 | georgetown
Processing Record 14 of Set 1 | bethel
Processing Record 15 of Set 1 | dudinka
Processing Record 16 of Set 1 | ponta delga

In [5]:
city_data_df.head(10)

,City,Country,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Current Description
0,Waitangi,NZ,-43.9535,-176.5597,60.82,83,100,1.01,overcast clouds
1,Andergrove,AU,-21.0833,149.1833,75.00,81,82,19.62,broken clouds
2,Port-Aux-Francais,TF,-49.3500,70.2167,46.42,88,100,32.66,overcast clouds
3,Praia Da Vitoria,PT,38.7333,-27.0667,59.25,77,75,3.44,broken clouds
4,Touros,BR,-5.1989,-35.4608,81.66,79,53,11.56,broken clouds
5,Sittwe,MM,20.1500,92.9000,77.86,88,46,6.26,scattered clouds
6,Saudarkrokur,IS,65.7461,-19.6394,27.00,80,100,1.72,overcast clouds
7,Tura,IN,25.5198,90.2201,74.48,73,88,8.25,light rain
8,Grytviken,GS,-54.2811,-36.5092,36.84,98,100,14.09,light rain
9,Afaahiti,PF,-17.7500,-149.2833,79.74,82,83,18.63,light rain


In [6]:
# Confirm columns 
city_data_df.columns

Index(['City', 'Country', 'Lat', 'Lng', 'Max Temp', 'Humidity', 'Cloudiness',
       'Wind Speed', 'Current Description'],
      dtype='str')

---
## 2. Filter Destinations by Temperature

The user specifies a comfortable temperature window (°F). Only cities whose max temperature falls within this range are kept.

In [14]:
min_temp = float(input("Minimum temperature (°F) for your trip: "))
max_temp = float(input("Maximum temperature (°F) for your trip: "))
print(f"Filtering cities with max temperatures between {min_temp}°F and {max_temp}°F...")

Filtering cities with max temperatures between 30.0°F and 120.0°F...


In [15]:
preferred_cities_df = city_data_df.loc[
    (city_data_df["Max Temp"] >= min_temp) &
    (city_data_df["Max Temp"] <= max_temp)
]

# aggressively drop rows with missing values
preferred_cities_df = preferred_cities_df.dropna()

print(f"{len(preferred_cities_df)} cities match the {min_temp}-{max_temp} degreesF range.")
preferred_cities_df.head(10)

645 cities match the 30.0-120.0 degreesF range.


,City,Country,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Current Description
0,Waitangi,NZ,-43.9535,-176.5597,60.82,83,100,1.01,overcast clouds
1,Andergrove,AU,-21.0833,149.1833,75.00,81,82,19.62,broken clouds
2,Port-Aux-Francais,TF,-49.3500,70.2167,46.42,88,100,32.66,overcast clouds
3,Praia Da Vitoria,PT,38.7333,-27.0667,59.25,77,75,3.44,broken clouds
4,Touros,BR,-5.1989,-35.4608,81.66,79,53,11.56,broken clouds
5,Sittwe,MM,20.1500,92.9000,77.86,88,46,6.26,scattered clouds
7,Tura,IN,25.5198,90.2201,74.48,73,88,8.25,light rain
8,Grytviken,GS,-54.2811,-36.5092,36.84,98,100,14.09,light rain
9,Afaahiti,PF,-17.7500,-149.2833,79.74,82,83,18.63,light rain
10,Badger,US,64.8000,-147.5333,32.38,44,0,3.44,clear sky


## 3. Find Nearby Hotels via Geoapify

For each candidate city, we search for hotels within a 5 km radius using the Geoapify Places API. Cities where no hotel is found are dropped from the final results.

In [16]:
hotel_df = preferred_cities_df[
    ["City", "Country", "Max Temp", "Current Description", "Lat", "Lng"]
].copy()
hotel_df["Hotel Name"] = ""

# Geoapify search parameters
radius = 5000 # search radius in meters
params = {
    "categories": "accommodation.hotel",
    "apiKey": geoapify_key,
    "limit": 20
}
base_url = "https://api.geoapify.com/v2/places"

print("Starting hotel search...")

for index, row in hotel_df.iterrows():
    params["filter"] = f"circle:{row['Lng']},{row['Lat']},{radius}"
    params["bias"] = f"proximity:{row['Lng']},{row['Lat']}"

    try:
        response = requests.get(base_url, params=params)
        results = response.json()
        hotel_df.loc[index, "Hotel Name"] = results["features"][0]["properties"]["name"]
    except (requests.exceptions.RequestException, KeyError, IndexError):
        hotel_df.loc[index, "Hotel Name"] = "No hotel found"

    print(f"  {row['City']} - nearest hotel:  {hotel_df.loc[index, 'Hotel Name']}")

# Remove cities with no hotel match
clean_hotel_df = hotel_df.loc[hotel_df["Hotel Name"] != "No hotel found"]
print(f"\nHotel search complete  - {len(clean_hotel_df)} destinations with hotels.")

Starting hotel search...
  Waitangi - nearest hotel:  Hotel Chathams
  Andergrove - nearest hotel:  No hotel found
  Port-Aux-Francais - nearest hotel:  Keravel
  Praia Da Vitoria - nearest hotel:  Salles
  Touros - nearest hotel:  Pousada Atlântico
  Sittwe - nearest hotel:  Yuzana Aung Motel 1
  Tura - nearest hotel:  Hotel Polo Orchid
  Grytviken - nearest hotel:  No hotel found
  Afaahiti - nearest hotel:  Omati Lodge
  Badger - nearest hotel:  No hotel found
  Georgetown - nearest hotel:  Page 63 hostel
  Bethel - nearest hotel:  Hampton Inn Danbury
  Ponta Delgada - nearest hotel:  Vila Galé Collection S. Miguel
  Kingston - nearest hotel:  Aphrodite Hotel
  Lorengau - nearest hotel:  Seeadler Bay Hotel
  Kokopo - nearest hotel:  Hotel Kokopo
  Dogondoutchi - nearest hotel:  Maison Alheri
  Queenstown - nearest hotel:  Queens Hotel
  Hadibu - nearest hotel:  No hotel found
  Mil'Kovo - nearest hotel:  Дрозды
  Maintirano - nearest hotel:  Hotel Safari
  Kudahuvadhoo - nearest hot

In [17]:
clean_hotel_df.head(10)

,City,Country,Max Temp,Current Description,Lat,Lng,Hotel Name
0,Waitangi,NZ,60.82,overcast clouds,-43.9535,-176.5597,Hotel Chathams
2,Port-Aux-Francais,TF,46.42,overcast clouds,-49.3500,70.2167,Keravel
3,Praia Da Vitoria,PT,59.25,broken clouds,38.7333,-27.0667,Salles
4,Touros,BR,81.66,broken clouds,-5.1989,-35.4608,Pousada Atlântico
5,Sittwe,MM,77.86,scattered clouds,20.1500,92.9000,Yuzana Aung Motel 1
7,Tura,IN,74.48,light rain,25.5198,90.2201,Hotel Polo Orchid
9,Afaahiti,PF,79.74,light rain,-17.7500,-149.2833,Omati Lodge
11,Georgetown,MY,80.53,few clouds,5.4112,100.3354,Page 63 hostel
12,Bethel,US,52.74,overcast clouds,41.3712,-73.4140,Hampton Inn Danbury
14,Ponta Delgada,PT,60.19,broken clouds,37.7333,-25.6667,Vila Galé Collection S. Miguel


### Destination Map

Each point represents a candidate city. Point size corresponds to temperature; hover for hotel and weather details.

In [39]:
hotel_map = clean_hotel_df.hvplot.points(
    "Lng", "Lat",
    geo=True,
    tiles="OSM",
    frame_width=700,
    frame_height=500,
    size="Max Temp",
    scale=1,
    color="City",
    hover_cols=["Hotel Name", "Country", "Current Description"]
)

hotel_map

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (City,Max Temp,Hotel Name,Country,Current Description)

In [22]:
# Save clean hotel data to CSV
clean_hotel_df.to_csv("Weather_Database/Preferred_Cities_with_Hotels.csv", index=False)
print("Clean hotel data saved to Weather_Database/Preferred_Cities_with_Hotels.csv")

Clean hotel data saved to Weather_Database/Preferred_Cities_with_Hotels.csv


---
## 4. Plan the Driving Itinerary

Four cities are chosen for a round-trip route. The trip starts and ends in El Granada, with stops in Laguna, Clearlake, and Fortuna along the way. The waypoints are passed to the Geoapify Routing API, which returns the full driving route geometry.

In [ ]:
# If needed, load the clean hotel data from CSV
# clean_hotel_df = pd.read_csv("Weather_Database/Preferred_Cities_with_Hotels.csv")

In [29]:
# Define the round-trip stops
vacation_start = clean_hotel_df.loc[clean_hotel_df["City"] == "El Granada"]
vacation_end   = clean_hotel_df.loc[clean_hotel_df["City"] == "El Granada"]
vacation_stop1 = clean_hotel_df.loc[clean_hotel_df["City"] == "Laguna"]
vacation_stop2 = clean_hotel_df.loc[clean_hotel_df["City"] == "Clearlake"]
vacation_stop3 = clean_hotel_df.loc[clean_hotel_df["City"] == "Fortuna"]

itinerary_df = pd.concat([
    vacation_start, vacation_stop1, vacation_stop2, vacation_stop3, vacation_end
])

print(f"Itinerary: {' > '.join(itinerary_df['City'].values)}")

Itinerary: El Granada > Laguna > Clearlake > Fortuna > El Granada


In [30]:
itinerary_df

,City,Country,Max Temp,Current Description,Lat,Lng,Hotel Name
494,El Granada,US,77.52,clear sky,37.5027,-122.4694,Beach House
36,Laguna,US,81.03,clear sky,38.4210,-121.4238,Holiday Inn Express & Suites
214,Clearlake,US,78.82,overcast clouds,38.9582,-122.6264,Best Western El Grande Inn
221,Fortuna,US,72.93,clear sky,40.5982,-124.1573,Comfort Inn & Suites Redwood Country
494,El Granada,US,77.52,clear sky,37.5027,-122.4694,Beach House


In [31]:
# Build the pipe-delimited waypoints string expected by Geoapify
waypoints_df = itinerary_df[["Lat", "Lng"]].copy()

waypoints_str = "|".join(
    f"{row['Lat']},{row['Lng']}" for _, row in waypoints_df.iterrows()
)

params = {
    "mode": "drive",
    "apiKey": geoapify_key,
    "waypoints": waypoints_str
}

try:
    response = requests.get("https://api.geoapify.com/v1/routing", params=params)
    response.raise_for_status()
    route_response = response.json()
    print("Route retrieved successfully.")
except requests.exceptions.RequestException as e:
    print(f"Routing API error: {e}")

Route retrieved successfully.


### Route Map

The driving route is overlaid on the base map using GeoViews' `Path` function, composited with the city point markers.

In [32]:
# Parse the route legs into flat lists of lon/lat
legs = route_response["features"][0]["geometry"]["coordinates"]

longitude_steps = []
latitude_steps = []

for leg in legs:
    for coord in leg:
        longitude_steps.append(coord[0])
        latitude_steps.append(coord[1])

route_df = pd.DataFrame({
    "longitude": longitude_steps,
    "latitude": latitude_steps
})

print(f"Route contains {len(route_df)} coordinate points.")

Route contains 20339 coordinate points.


In [40]:
# Route line
route_path = gv.Path(route_df)

# Route points layer
route_map = route_df.hvplot.points(
    "longitude", "latitude",
    geo=True,
    tiles="CartoLight",
    frame_width=700,
    frame_height=500,
    color="blue"
)

# Composite map
route_map * route_path

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [longitude,latitude]
   .Path.I   :Path   [longitude,latitude]